In [5]:
import chess
from chess import Move, Board, svg
import keras.models
import numpy as np
import os, sys
import tensorflow as tf
module_path = os.path.abspath(os.path.join('.', 'utils'))
if module_path not in sys.path:
    sys.path.append(module_path)
from utils2 import DataGenerator, plot_graph, bar_plot, two_bar_plot, StepwiseLogger, decode_pos

In [2]:
model = keras.models.load_model("./stored_models/model_1_black_pieces.h5")
model.summary()

Model: "model_1"
__________________________________________________________________________________________________
 Layer (type)                   Output Shape         Param #     Connected to                     
 input_2 (InputLayer)           [(None, 8, 8, 7)]    0           []                               
                                                                                                  
 conv2d_9 (Conv2D)              (None, 8, 8, 64)     4096        ['input_2[0][0]']                
                                                                                                  
 batch_normalization_16 (BatchN  (None, 8, 8, 64)    256         ['conv2d_9[0][0]']               
 ormalization)                                                                                    
                                                                                                  
 leaky_re_lu_16 (LeakyReLU)     (None, 8, 8, 64)     0           ['batch_normalization_16[0]

In [9]:
board = chess.Board()

In [19]:
m = 'e2e4'

In [23]:
move = board.parse_uci(m)
move

Move.from_uci('e2e4')

In [26]:
board.fen()

'rnbqkbnr/pppppppp/8/8/8/8/PPPPPPPP/RNBQKBNR w KQkq - 0 1'

In [10]:
class ModelSingleton:
    _instance = None
    _model = None

    def __new__(cls, *args, **kwargs):
        if cls._instance is None:
            cls._instance = super(ModelSingleton, cls).__new__(cls)
            gpus = tf.config.experimental.list_physical_devices('GPU')
            if gpus:
                try:
                    for gpu in gpus:
                        tf.config.experimental.set_memory_growth(gpu, True)
                except RuntimeError as e:
                    print(e)
            cls._model = keras.models.load_model("./stored_models/model_1_black_pieces.h5")
        return cls._instance 
    
    def get_model(self):
        return self._model

In [9]:
ModelSingleton._instance = None

In [3]:
class SharedModelEngine:
    def __init__(self):
        # Get the shared model from ModelSingleton
        self.engine = ModelSingleton().get_model()

    def predict(self, input_data):
        # Ensure input_data is in the correct format for your model
        return self.engine.predict(np.array([input_data]))

In [19]:
instance1 = SharedModelEngine()
instance2 = SharedModelEngine()

input_data1 = np.random.rand(8, 8, 7)  # Example input shape
input_data2 = np.random.rand(8, 8, 7)  # Example input shape
instance1.predict(input_data1)
instance2.predict(input_data2)

1/1 [==============================] - 0s 16ms/step


[array([[0.0000000e+00, 9.9712352e-31, 5.0149060e-30, ..., 1.2116994e-22,
         4.4953807e-24, 0.0000000e+00]], dtype=float32),
 array([[6.76305825e-03, 3.30891652e-07, 1.35197666e-13, 2.11257227e-02,
         1.07803244e-04, 4.89217611e-10, 1.44124782e-10, 2.89798249e-03,
         2.77998596e-10, 1.64332360e-01, 3.62534794e-11, 8.56638198e-12,
         2.89903469e-02, 2.89255348e-14, 4.60090493e-10, 1.40341003e-11,
         7.86785847e-07, 8.74187736e-22, 1.08112372e-18, 1.13628690e-10,
         2.38799751e-02, 1.83719134e-11, 5.41176045e-13, 7.35422873e-18,
         1.17437663e-02, 2.61454983e-03, 2.34273875e-06, 2.65446976e-02,
         3.27681975e-16, 5.19731575e-17, 6.61262630e-29, 1.95902539e-05,
         3.04893483e-21, 9.81487400e-15, 6.72234818e-02, 3.15755635e-01,
         2.16446742e-02, 4.97470731e-10, 3.40524539e-02, 2.76821537e-16,
         4.08671449e-22, 2.13780712e-11, 5.38883478e-12, 1.16560973e-01,
         6.22242173e-22, 1.28100410e-01, 3.76071787e-14, 4.7738766

In [ ]:
board.push

In [15]:
list(board.legal_moves)[0]

Move.from_uci('g1h3')

In [27]:
legal_moves = [move.uci()[:4] for move in board.legal_moves]
legal_moves

['g1h3',
 'g1f3',
 'b1c3',
 'b1a3',
 'h2h3',
 'g2g3',
 'f2f3',
 'e2e3',
 'd2d3',
 'c2c3',
 'b2b3',
 'a2a3',
 'h2h4',
 'g2g4',
 'f2f4',
 'e2e4',
 'd2d4',
 'c2c4',
 'b2b4',
 'a2a4']

In [29]:
def encode_sparse_single_move(moves):
    column_to_number = {
        'a': 1,
        'b': 2,
        'c': 3,
        'd': 4,
        'e': 5,
        'f': 6,
        'g': 7,
        'h': 8
    }
    encoded_moves = []
    for move in moves:
        encoded_moves.append((((column_to_number.get(move[0]) - 1)) + ((8 - int(move[1])) * 8)) * 64 + (((column_to_number.get(move[2]) - 1)) + ((8 - int(move[3])) * 8)))
    return np.array(encoded_moves)

In [35]:
encoded_legal_moves = encode_sparse_single_move(legal_moves)
encoded_legal_moves

array([4015, 4013, 3690, 3688, 3567, 3502, 3437, 3372, 3307, 3242, 3177,
       3112, 3559, 3494, 3429, 3364, 3299, 3234, 3169, 3104])

In [34]:
ones = np.ones(4096)
ones

array([1., 1., 1., ..., 1., 1., 1.])

In [38]:
legal_best_moves = [0 if i not in encoded_legal_moves else val for i, val in enumerate(ones)]
legal_best_moves

[0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,


In [39]:
sum(legal_best_moves)

20.0

In [3]:
piece_to_one_hot = {
    'r':[-1.,0.,0.,0.,0.,0.],
    'n':[0.,-1.,0.,0.,0.,0.],
    'b':[0.,0.,-1.,0.,0.,0.],
    'q':[0.,0.,0.,-1.,0.,0.],
    'k':[0.,0.,0.,0.,-1.,0.],
    'p':[0.,0.,0.,0.,0.,-1.],
    'R':[1.,0.,0.,0.,0.,0.],
    'N':[0.,1.,0.,0.,0.,0.],
    'B':[0.,0.,1.,0.,0.,0.],
    'Q':[0.,0.,0.,1.,0.,0.],
    'K':[0.,0.,0.,0.,1.,0.],
    'P':[0.,0.,0.,0.,0.,1.]
}
empty_square = [0.,0.,0.,0.,0.,0.]

def get_one_hot(board: str):
    one_hot_board = []
    rows = board.split(sep='/')
    for row in rows:
        one_hot_row = []
        for piece in [*row]:
            if piece.isdigit():
                for i in range(int(piece)):
                    one_hot_row.append(empty_square)
            else:
                one_hot_row.append(piece_to_one_hot[piece])
        one_hot_board.append(one_hot_row)
    return one_hot_board

def decode_single_move(movements):
    decoded_movements = []
    for move in movements:
        encoded_initial_pos = decode_pos([move // 64])[0]
        encoded_final_pos = decode_pos([move % 64])[0]
        decoded_movements.append(encoded_initial_pos + encoded_final_pos)
        
    return np.array(decoded_movements)

def get_best_legal_move(predictions, board):
    legal_moves = [move.uci()[:4] for move in board.legal_moves]
    
    return np.argmax(predictions)

def get_move(model, board):
    one_hot_board = get_one_hot(board.board_fen())
    predicted_model_ouput = model(np.array([one_hot_board]))[0]
    best_move = get_best_legal_move(predicted_model_ouput, board)
    return decode_single_move([best_move])[0]

